# Non-Speech Pool — Whisper Transcription

Resumable Whisper transcription over the **non-speech** subset of udio_speech_labels.csv (Notebook 1's output: is_speech == False). Nothing is re-downloaded — audio is read from Notebook 1's local data/audio/ cache.

1. **Transcribe** — run [aster-whisper](https://github.com/SYSTRAN/faster-whisper) (large-v3 by default) on every non-speech clip's cached audio, and append the transcript (plus detected language, duration, segment count) to data/audio_non_speech_transcriptions.csv. udio_speech_labels.csv itself is never modified.

**Device notes:**
- aster-whisper runs on CTranslate2, which supports **CUDA or CPU only** — there's no Apple Metal/MPS backend, so on a Mac this stage falls back to CPU (int8 compute type) and will be slow for large-v3. On a CUDA box it auto-selects loat16. Override via .env (WHISPER_DEVICE, WHISPER_COMPUTE_TYPE) if needed.

**Dependencies not yet in 
equirements.txt:** aster-whisper. Install with the cell below (mirrors how Notebook 4 installs packages ad hoc).

Structure mirrors Notebook 4's Stage 1 (transcription only).


## 0. Install dependencies

Run once in the same .venv used for the other notebooks. Restart the kernel if VS Code asks you to, then continue from the next cell.


In [ ]:
%pip install -U faster-whisper

In [ ]:
import os
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from faster_whisper import WhisperModel
from tqdm import tqdm

load_dotenv()


## Configuration

All values come from .env at the repo root, with sensible fallbacks. Device/compute-type are auto-detected unless explicitly overridden.


In [ ]:
# Locations (relative to this notebook's directory, i.e. jupyter_notebooks/)
DATA_DIR = Path(os.getenv("DATA_DIR", "../data"))
AUDIO_DIR = DATA_DIR / "audio"

# Input: Notebook 1's output. Only rows with is_speech == False are processed.
SPEECH_LABELS_CSV = DATA_DIR / os.getenv("SPEECH_LABELS_CSV", "audio_speech_labels.csv")

# Output: brand-new CSV for non-speech transcripts. Neither SPEECH_LABELS_CSV
# nor audio_transcriptions.csv (speech) is modified by this notebook.
TRANSCRIPTS_CSV = DATA_DIR / os.getenv(
    "NON_SPEECH_TRANSCRIPTS_CSV",
    "audio_non_speech_transcriptions.csv",
)

ID_COLUMN = os.getenv("ID_COLUMN", "id")
AUDIO_URL_COLUMN = os.getenv("AUDIO_URL_COLUMN", "streamableUrl")
SPEECH_COLUMN = "is_speech"

SAMPLE_RATE = 16_000  # both faster-whisper and the cached-audio decoder expect 16kHz mono.

# Save progress every N newly processed rows.
SAVE_EVERY = int(os.getenv("SAVE_EVERY", "10"))

# Maximum time allowed for ffmpeg to decode one local cached file.
FFMPEG_TIMEOUT_SECONDS = int(os.getenv("FFMPEG_TIMEOUT_SECONDS", "120"))

SUPPORTED_AUDIO_SUFFIXES = {".mp3", ".m4a", ".wav", ".ogg", ".aac", ".flac"}


# --- Whisper configuration ---

WHISPER_MODEL_SIZE = os.getenv("WHISPER_MODEL_SIZE", "large-v3")

# Leave unset to let Whisper auto-detect the spoken language.
_language_env = os.getenv("WHISPER_LANGUAGE", "").strip()
WHISPER_LANGUAGE = _language_env if _language_env else None

# Parallel inference streams inside the single, shared WhisperModel --
# what faster-whisper/CTranslate2 uses to safely serve transcribe() calls
# from multiple Python threads at once. Independent of MAX_WORKERS below.
WHISPER_NUM_WORKERS = int(os.getenv("WHISPER_NUM_WORKERS", "4"))

# Threads CTranslate2 uses per inference stream when running on CPU.
# 0 lets CTranslate2 pick automatically. Ignored on cuda.
WHISPER_CPU_THREADS = int(os.getenv("WHISPER_CPU_THREADS", "0"))

# Rows processed concurrently. Each one decodes cached audio via ffmpeg
# (CPU-bound) and then calls the shared Whisper model above. Keep this
# >= WHISPER_NUM_WORKERS so the model always has work queued.
MAX_WORKERS = int(os.getenv("TRANSCRIBE_MAX_WORKERS", "8"))


def pick_whisper_device_and_compute_type() -> tuple[str, str]:
    """CTranslate2 (faster-whisper's backend) only supports CUDA or CPU --
    there is no Apple MPS/Metal support, unlike plain PyTorch models."""
    if torch.cuda.is_available():
        return "cuda", "float16"
    return "cpu", "int8"


_auto_whisper_device, _auto_whisper_compute_type = pick_whisper_device_and_compute_type()
WHISPER_DEVICE = os.getenv("WHISPER_DEVICE", "").strip() or _auto_whisper_device
WHISPER_COMPUTE_TYPE = os.getenv("WHISPER_COMPUTE_TYPE", "").strip() or _auto_whisper_compute_type

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Whisper model: {WHISPER_MODEL_SIZE} ({WHISPER_DEVICE}, {WHISPER_COMPUTE_TYPE})")
print(f"Output CSV:    {TRANSCRIPTS_CSV}")


## Locate cached audio

Reuses Notebook 1's exact cache-path rule (extension derived from the streamable URL), with a fallback scan by ID for anything that doesn't match. No audio is downloaded here — everything is read from AUDIO_DIR.


In [ ]:
def cache_path_for(row_id: str, url: str) -> Path:
    """Same cache-path rule used by Notebook 1."""
    suffix = Path(urlparse(url).path).suffix.lower() or ".mp3"
    if suffix not in SUPPORTED_AUDIO_SUFFIXES:
        suffix = ".mp3"
    return AUDIO_DIR / f"{row_id}{suffix}"


def find_cached_audio(row: dict) -> Path | None:
    """Find the audio downloaded by Notebook 1, or None if it isn't cached."""
    row_id = str(row.get(ID_COLUMN)).strip()
    url_value = row.get(AUDIO_URL_COLUMN)

    if pd.notna(url_value) and str(url_value).strip():
        expected_path = cache_path_for(row_id, str(url_value).strip())
        if expected_path.exists() and expected_path.stat().st_size > 0:
            return expected_path

    fallback_candidates = [
        candidate
        for suffix in SUPPORTED_AUDIO_SUFFIXES
        if (candidate := AUDIO_DIR / f"{row_id}{suffix}").exists() and candidate.stat().st_size > 0
    ]

    if not fallback_candidates:
        return None

    # Prefer the largest complete cached file if multiple formats exist.
    return max(fallback_candidates, key=lambda path: path.stat().st_size)


def decode_audio_for_whisper(audio_path: Path, sample_rate: int = SAMPLE_RATE) -> np.ndarray:
    """Decode a local audio file into mono float32 PCM samples at 16kHz."""
    command = [
        "ffmpeg",
        "-nostdin",
        "-hide_banner",
        "-loglevel", "error",
        "-i", str(audio_path),
        "-vn",
        "-ac", "1",
        "-ar", str(sample_rate),
        "-f", "f32le",
        "-",
    ]

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=FFMPEG_TIMEOUT_SECONDS,
        )
    except subprocess.TimeoutExpired as error:
        raise TimeoutError(f"ffmpeg timed out while decoding {audio_path.name}") from error

    if result.returncode != 0:
        error_message = result.stderr.decode("utf-8", errors="ignore").strip()
        raise RuntimeError(error_message or f"ffmpeg failed to decode {audio_path.name}")

    waveform = np.frombuffer(result.stdout, dtype=np.float32)
    waveform = np.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0)

    if waveform.size == 0:
        raise ValueError(f"Decoded waveform is empty: {audio_path.name}")

    return waveform


## Transcribe non-speech clips with aster-whisper

A single WhisperModel instance is built once (before any worker threads start) and shared across all of them — faster-whisper/CTranslate2 is explicitly designed to be called concurrently this way via 
um_workers, so this gets parallelism without paying for N separate model copies in memory. MAX_WORKERS mostly controls how many ffmpeg decodes run at once so the model always has queued work.


In [ ]:
def build_whisper_model() -> WhisperModel:
    return WhisperModel(
        WHISPER_MODEL_SIZE,
        device=WHISPER_DEVICE,
        compute_type=WHISPER_COMPUTE_TYPE,
        cpu_threads=WHISPER_CPU_THREADS,
        num_workers=WHISPER_NUM_WORKERS,
    )


def transcribe_audio(whisper_model: WhisperModel, audio_path: Path) -> dict:
    waveform = decode_audio_for_whisper(audio_path)

    segments, info = whisper_model.transcribe(
        waveform,
        language=WHISPER_LANGUAGE,
    )

    segments = list(segments)
    transcript = " ".join(segment.text.strip() for segment in segments).strip()

    return {
        "transcript": transcript,
        "language": info.language,
        "language_probability": info.language_probability,
        "audio_duration_seconds": info.duration,
        "segment_count": len(segments),
    }


def transcribe_row(whisper_model: WhisperModel, row: dict) -> dict:
    """Transcribe one row's cached audio. Returns the DB row + transcript fields."""
    result = dict(row)
    result.update({
        "transcript": None,
        "language": None,
        "language_probability": None,
        "audio_duration_seconds": None,
        "segment_count": None,
        "transcription_status": None,
        "transcription_error": None,
    })

    row_id = row.get(ID_COLUMN)

    try:
        audio_path = find_cached_audio(row)
        if audio_path is None:
            raise FileNotFoundError("Audio not downloaded (see Notebook 1, Stage 2).")

        result.update(transcribe_audio(whisper_model, audio_path))
        result["transcription_status"] = "completed"

    except Exception as error:
        result["transcription_status"] = "failed"
        result["transcription_error"] = str(error)
        print(f"[TRANSCRIBE FAILED] {ID_COLUMN}={row_id}: {error}")

    return result


In [ ]:
def load_existing_results(output_csv: Path) -> pd.DataFrame:
    if not output_csv.exists():
        return pd.DataFrame()

    try:
        existing = pd.read_csv(output_csv)
        if ID_COLUMN in existing.columns:
            existing[ID_COLUMN] = existing[ID_COLUMN].astype(str)
        return existing
    except Exception as error:
        print(f"Could not read existing CSV {output_csv}: {error}")
        return pd.DataFrame()


def get_processed_ids(existing_results: pd.DataFrame) -> set:
    if existing_results.empty or ID_COLUMN not in existing_results.columns:
        return set()
    return set(existing_results[ID_COLUMN].astype(str).tolist())


def save_results(existing_results: pd.DataFrame, new_results: list, output_csv: Path) -> pd.DataFrame:
    new_dataframe = pd.DataFrame(new_results)

    if existing_results.empty:
        combined_dataframe = new_dataframe
    else:
        combined_dataframe = pd.concat([existing_results, new_dataframe], ignore_index=True)

    combined_dataframe.to_csv(output_csv, index=False)
    return combined_dataframe


In [ ]:
if not SPEECH_LABELS_CSV.exists():
    raise FileNotFoundError(
        f"Could not find: {SPEECH_LABELS_CSV}\n"
        "Run Notebook 1 through Stage 3 first so that audio_speech_labels.csv exists."
    )

speech_labels = pd.read_csv(SPEECH_LABELS_CSV)
speech_labels[ID_COLUMN] = speech_labels[ID_COLUMN].astype(str)

if SPEECH_COLUMN not in speech_labels.columns:
    raise KeyError(f"Expected Notebook 1's '{SPEECH_COLUMN}' column, but it was not found.")

# Non-speech pool: invert the speech filter from Notebook 4.
non_speech_rows = speech_labels[speech_labels[SPEECH_COLUMN] == False]

existing_transcripts = load_existing_results(TRANSCRIPTS_CSV)
processed_ids = get_processed_ids(existing_transcripts)

rows_to_transcribe = non_speech_rows[~non_speech_rows[ID_COLUMN].isin(processed_ids)]

print(f"Non-speech-labeled rows: {len(non_speech_rows)}")
print(f"Already transcribed:     {len(processed_ids)}")
print(f"Remaining rows:          {len(rows_to_transcribe)}")
print(f"Download workers:        {MAX_WORKERS}")
print(f"Inference workers:       {WHISPER_NUM_WORKERS}")

whisper_model = build_whisper_model()

new_transcripts = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(transcribe_row, whisper_model, row.to_dict())
        for _, row in rows_to_transcribe.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Transcribing non-speech", unit="clip"):
        new_transcripts.append(future.result())

        if len(new_transcripts) % SAVE_EVERY == 0:
            existing_transcripts = save_results(existing_transcripts, new_transcripts, TRANSCRIPTS_CSV)
            new_transcripts = []

if new_transcripts:
    existing_transcripts = save_results(existing_transcripts, new_transcripts, TRANSCRIPTS_CSV)

print("\nTranscription stage finished.")
print(f"Results saved to: {TRANSCRIPTS_CSV}")


## Verify


In [ ]:
from IPython.display import display

result_transcripts = pd.read_csv(TRANSCRIPTS_CSV) if TRANSCRIPTS_CSV.exists() else pd.DataFrame()

if result_transcripts.empty:
    print("No transcription results yet.")
else:
    print("Transcription status:")
    print(result_transcripts["transcription_status"].value_counts(dropna=False))
    print()

    completed = result_transcripts[result_transcripts["transcription_status"] == "completed"].copy()
    if not completed.empty:
        completed["transcript"] = completed["transcript"].fillna("").astype(str).str.strip()
        empty_rate = (completed["transcript"] == "").mean()
        print(f"Completed rows: {len(completed)}")
        print(f"Empty-transcript rate: {empty_rate:.1%}")
        print()

    display(result_transcripts[[ID_COLUMN, "transcription_status", "language", "segment_count", "transcript"]].head(10))


## Explore - sample non-speech clips

Play a few completed transcriptions so you can hear what Whisper produced on VAD-labeled non-speech audio.


In [ ]:
import glob
import random
from IPython.display import display, Audio

if not TRANSCRIPTS_CSV.exists():
    raise FileNotFoundError(f"Missing {TRANSCRIPTS_CSV}. Run the transcription stage first.")

transcripts_df = pd.read_csv(TRANSCRIPTS_CSV)
transcripts_df[ID_COLUMN] = transcripts_df[ID_COLUMN].astype(str)
completed = transcripts_df[transcripts_df["transcription_status"] == "completed"].copy()

audio_files = glob.glob(os.path.join(str(AUDIO_DIR), "*.*"))
audio_path_map = {os.path.splitext(os.path.basename(f))[0]: f for f in audio_files}

print(f"Completed non-speech transcripts: {len(completed)}")
print(f"Audio files found:                {len(audio_path_map)}")

NUM_SAMPLES = 5
random.seed(42)
sample_n = min(NUM_SAMPLES, len(completed))
if sample_n == 0:
    print("No completed rows to sample.")
else:
    for _, row in completed.sample(n=sample_n, random_state=42).iterrows():
        clip_id = row[ID_COLUMN]
        transcript = str(row.get("transcript") or "").strip()
        lang = row.get("language", "?")
        print(f"\n{'='*70}")
        print(f"id={clip_id}  lang={lang}  segments={row.get('segment_count')}")
        print(f"Transcript: {transcript if transcript else '(empty)'}")
        path = audio_path_map.get(clip_id)
        if path:
            display(Audio(path))
        else:
            print("(audio file not found in cache)")
